# 17. Supervised Learning: K-Nearest Neighbors (K-NN)

## Algorithm Category
**Type**: Supervised Learning - Classification/Regression  
**Complexity**: Low-Medium  
**Use Case**: Instance-based learning using local similarity

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the K-NN algorithm and its lazy learning approach
- Implement K-NN for both classification and regression
- Choose optimal k value using cross-validation
- Understand distance metrics (Euclidean, Manhattan, Minkowski)
- Handle the curse of dimensionality
- Apply K-NN to real-world problems

## Historical Context

K-Nearest Neighbors is one of the simplest machine learning algorithms:
- First proposed in 1951 by Evelyn Fix and Joseph Hodges
- Non-parametric method (makes no assumptions about data distribution)
- Instance-based or lazy learning (no explicit training phase)

**Key Papers/References:**
- Fix, E. & Hodges, J.L. (1951). "Discriminatory analysis, nonparametric discrimination"
- Cover, T. & Hart, P. (1967). "Nearest neighbor pattern classification"

## When to Use K-Nearest Neighbors

K-NN is appropriate when:
- You have a small to medium dataset
- Data has local structure (similar instances are close together)
- You need a simple, interpretable baseline
- Non-parametric approach is preferred
- Real-time prediction is not critical (can be slow for large datasets)
- Working with numerical features

## Theory & Mechanics

### Mathematical Foundation

K-NN makes predictions based on the k nearest training examples.

**Euclidean Distance:**
$$d(x_i, x_j) = \sqrt{\sum_{m=1}^{n}(x_{im} - x_{jm})^2}$$

**Manhattan Distance:**
$$d(x_i, x_j) = \sum_{m=1}^{n}|x_{im} - x_{jm}|$$

**Minkowski Distance (general form):**
$$d(x_i, x_j) = \left(\sum_{m=1}^{n}|x_{im} - x_{jm}|^p\right)^{1/p}$$

**Classification (majority vote):**
$$\hat{y} = \text{mode}(\{y_i : x_i \in N_k(x)\})$$

**Regression (mean):**
$$\hat{y} = \frac{1}{k}\sum_{x_i \in N_k(x)} y_i$$

Where $N_k(x)$ is the set of k nearest neighbors to $x$.

### How It Works

1. **Training**: Store all training examples (lazy learning - no model building)
2. **Prediction**: 
   - Calculate distances to all training examples
   - Find k nearest neighbors
   - For classification: majority vote
   - For regression: average of neighbors' values
3. **Weighted K-NN**: Can weight neighbors by inverse distance

### Key Hyperparameters

- **n_neighbors (k)**: Number of neighbors to consider (most important)
- **weights**: 'uniform' (equal weight) or 'distance' (inverse distance weighting)
- **metric**: Distance metric ('euclidean', 'manhattan', 'minkowski', etc.)
- **algorithm**: Algorithm for finding neighbors ('ball_tree', 'kd_tree', 'brute', 'auto')

### Limitations

- Slow prediction for large datasets (must compute distances to all training examples)
- Sensitive to irrelevant features (curse of dimensionality)
- Sensitive to local structure of data
- Requires feature scaling
- Memory intensive (stores all training data)


## Implementation

Let's implement K-NN for both classification and regression.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_iris,  # Iris flower classification dataset
    load_diabetes,  # Diabetes regression dataset
    make_classification  # Generate synthetic classification data
)
from sklearn.neighbors import (
    KNeighborsClassifier,  # K-NN for classification
    KNeighborsRegressor  # K-NN for regression
)
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score,  # Cross-validation scoring
    GridSearchCV  # Hyperparameter tuning
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy (for classification)
    mean_squared_error  # Calculate MSE (for regression)
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import (
    split_data,  # Split data into train/test sets
    evaluate_classifier,  # Evaluate classification models
    evaluate_regressor  # Evaluate regression models
)
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix  # Visualize confusion matrix
)
from src.processing.preprocessing import scale_features  # Normalize features
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# load_iris() loads the Iris flower dataset from scikit-learn
# This is a multiclass classification problem: predict flower species from measurements
iris = load_iris()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Flower measurements
# iris.data contains feature values (150 samples × 4 features)
# We convert to DataFrame for easier manipulation
X = pd.DataFrame(iris.data, columns=iris.feature_names)
# Features: sepal length, sepal width, petal length, petal width (4 measurements)

# y = Target (output): Flower species (what we want to predict)
# iris.target contains class labels (0, 1, or 2 for each sample)
y = pd.Series(iris.target, name='Species')
# 0 = setosa, 1 = versicolor, 2 = virginica

print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Classes: {iris.target_names.tolist()}")  # Output: ['setosa', 'versicolor', 'virginica']

# ============================================
# FEATURE SCALING: Critical for Distance-Based Algorithms
# ============================================

# K-NN is a distance-based algorithm - it uses distances to find nearest neighbors
# Features on different scales distort distance calculations
# Example: If one feature is in meters (0-10) and another in millimeters (0-10000),
#   the millimeter feature will dominate distance calculations
# Scaling ensures all features contribute equally to distance

# scale_features() normalizes features to have mean=0 and std=1
# fit=True means "learn scaling from this data" (use for training data)
X_scaled, scaler = scale_features(X, fit=True)
# X_scaled: Features normalized (mean=0, std=1 for each column)
# scaler: The scaling object (needed to scale test data with same transformation)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# CRITICAL: Split AFTER scaling to prevent data leakage
# If we split first, test data statistics could influence scaling

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# X_train: 120 samples for training (scaled features)
# X_test: 30 samples for testing
# y_train: Labels for training samples
# y_test: Labels for test samples (ground truth)


In [ ]:
# ============================================
# MODEL CREATION: K-Nearest Neighbors Classifier
# ============================================

# Create a KNeighborsClassifier model
# K-NN is a "lazy learner" - it doesn't build a model during training
# Instead, it stores all training data and uses it during prediction

# KNeighborsClassifier parameters:
# n_neighbors=5: Number of neighbors to consider (k value)
#   - This is the most important hyperparameter
#   - Smaller k (e.g., 1) = more sensitive to noise, more complex boundaries
#   - Larger k (e.g., 20) = smoother boundaries, less sensitive to noise
#   - Must be odd for binary classification to avoid ties
#   - Typical values: 3, 5, 7, 9, 11
#
# weights='uniform': How to weight neighbors
#   - 'uniform': All neighbors have equal weight (majority vote)
#   - 'distance': Weight by inverse distance (closer neighbors count more)
#   - Distance weighting can improve performance
#
# metric='euclidean': Distance metric to use
#   - 'euclidean': Standard straight-line distance (default)
#   - 'manhattan': Sum of absolute differences (L1 distance)
#   - 'minkowski': Generalization of Euclidean and Manhattan (with p parameter)
#   - Other options: 'chebyshev', 'hamming', etc.
model = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean')

# ============================================
# MODEL TRAINING: Storing Training Data
# ============================================

# .fit() for K-NN doesn't actually "train" - it just stores the training data
# This is why K-NN is called "lazy learning" - no computation happens until prediction
# The algorithm will use this stored data to find nearest neighbors during prediction
model.fit(X_train, y_train)  # Store training data

print("Model trained successfully!")  # Confirm data stored
print(f"Number of neighbors (k): {model.n_neighbors}")  # Display k value

# Note: K-NN training is very fast (just storing data)
# But prediction can be slow for large datasets (must compute distances to all training examples)

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate with helper function
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Validation & Testing

Let's find the optimal k value and validate the model.


In [ ]:
# ============================================
# VALIDATION 1: Finding Optimal k Value
# ============================================

# k (n_neighbors) is the most important hyperparameter for K-NN
# We'll test different k values to find the best one
# Too small k = overfitting (sensitive to noise)
# Too large k = underfitting (smooths out patterns)

# Test k values from 1 to 30
k_range = range(1, 31)  # [1, 2, 3, ..., 30]
k_scores = []  # Store CV accuracy for each k

# Test each k value
for k in k_range:
    # Create K-NN with this k value
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Perform 5-fold cross-validation
    # cross_val_score() splits data, trains, tests, repeats 5 times
    scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
    # Returns array of 5 accuracy scores (one per fold)
    
    # Store average accuracy across folds
    k_scores.append(scores.mean())

# ============================================
# FINDING OPTIMAL K
# ============================================

# np.argmax() finds index of maximum value
# k_range[...] gets the k value at that index
optimal_k = k_range[np.argmax(k_scores)]
print(f"Optimal k: {optimal_k} with CV accuracy: {max(k_scores):.3f}")

# ============================================
# VISUALIZING K VS ACCURACY
# ============================================

# Create line plot showing how accuracy changes with k
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Plot k vs accuracy
# 'o-' means circle markers connected by lines
plt.plot(k_range, k_scores, 'o-')

# Draw vertical line at optimal k
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k={optimal_k}')
# color='r': Red line
# linestyle='--': Dashed line

# Label axes
plt.xlabel('Number of Neighbors (k)')  # X-axis: k value
plt.ylabel('Cross-Validation Accuracy')  # Y-axis: accuracy
plt.title('Finding Optimal k for K-NN')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Usually there's a "sweet spot" where accuracy is highest
# - Very small k (1-3): High variance, sensitive to noise
# - Very large k (20+): High bias, may underfit
# - Optimal k balances bias and variance


In [ ]:
# ============================================
# VALIDATION 2: Comparing Weighting Schemes
# ============================================

# K-NN can weight neighbors differently:
# - 'uniform': All neighbors have equal weight (majority vote)
# - 'distance': Weight by inverse distance (closer neighbors count more)
# Distance weighting often improves performance

# Create two models with different weighting schemes
model_uniform = KNeighborsClassifier(n_neighbors=optimal_k, weights='uniform')
model_distance = KNeighborsClassifier(n_neighbors=optimal_k, weights='distance')

# Evaluate both using cross-validation
uniform_scores = cross_val_score(model_uniform, X_scaled, y, cv=5, scoring='accuracy')
distance_scores = cross_val_score(model_distance, X_scaled, y, cv=5, scoring='accuracy')

print("Weighting Comparison:")
print(f"  Uniform weights: {uniform_scores.mean():.3f} (+/- {uniform_scores.std():.3f})")
print(f"  Distance weights: {distance_scores.mean():.3f} (+/- {distance_scores.std():.3f})")

# ============================================
# SELECTING BEST WEIGHTING SCHEME
# ============================================

# Choose weighting scheme with higher CV accuracy
best_weights = 'distance' if distance_scores.mean() > uniform_scores.mean() else 'uniform'
print(f"\nBest weighting: {best_weights}")

# ============================================
# TRAINING FINAL MODEL
# ============================================

# Train final model with optimal k and best weighting
final_model = KNeighborsClassifier(n_neighbors=optimal_k, weights=best_weights)
final_model.fit(X_train, y_train)  # Store training data

# Make predictions on test set
final_pred = final_model.predict(X_test)  # Predictions using final model

# Calculate test accuracy
final_accuracy = accuracy_score(y_test, final_pred)  # How well final model performs
print(f"Final test accuracy: {final_accuracy:.3f}")

# Interpretation:
# - Distance weighting gives more influence to closer neighbors
# - This often improves performance, especially when neighbors vary in distance
# - Final model uses both optimal k and best weighting scheme


In [ ]:
# ============================================
# VALIDATION 3: Checking Model Output Validity
# ============================================

# validate_model_output() checks if predictions are valid
# task_type='classification' tells validator this is classification
validation_result = validate_model_output(final_pred, y_test.values, task_type='classification')
# Returns dictionary with validation results

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: Predictions must be valid
assert validation_result['valid'], "Invalid predictions!"
# If predictions are invalid (wrong shape, wrong values, etc.), stop execution

# Check 2: Accuracy must be better than random guessing
# For 3-class classification, random = 1/3 ≈ 0.333
assert final_accuracy > 0.5, "Accuracy should be better than random!"
# If accuracy ≤ 0.5, model is no better than guessing

print("✓ Validation checks passed")  # All checks passed!


## Regression Example

Let's apply K-NN to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_scaled, _ = scale_features(X_reg, fit=True)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg_scaled, y_reg, test_size=0.2, random_state=42)

# Find optimal k for regression
k_range_reg = range(1, 21)
k_scores_reg = []

for k in k_range_reg:
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    scores = cross_val_score(knn_reg, X_reg_scaled, y_reg, cv=5, scoring='neg_mean_squared_error')
    k_scores_reg.append(-scores.mean())

optimal_k_reg = k_range_reg[np.argmin(k_scores_reg)]
print(f"Optimal k for regression: {optimal_k_reg} with CV RMSE: {np.sqrt(min(k_scores_reg)):.3f}")

# Train and evaluate
knn_reg = KNeighborsRegressor(n_neighbors=optimal_k_reg, weights='distance')
knn_reg.fit(X_reg_train, y_reg_train)
y_reg_pred = knn_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print(f"\nRegression Results:")
print(f"  RMSE: {rmse:.3f}")

# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.6)
plt.plot([y_reg_test.min(), y_reg_test.max()], 
         [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('K-NN Regression: Predicted vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Real-World Application

Let's compare different distance metrics and use GridSearchCV.


In [ ]:
# Compare distance metrics
metrics_to_test = ['euclidean', 'manhattan', 'minkowski']
metric_results = {}

for metric in metrics_to_test:
    knn = KNeighborsClassifier(n_neighbors=optimal_k, metric=metric)
    scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
    metric_results[metric] = scores.mean()
    print(f"{metric.capitalize()} distance: {scores.mean():.3f} (+/- {scores.std():.3f})")

best_metric = max(metric_results, key=metric_results.get)
print(f"\nBest metric: {best_metric}")

# Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")

best_knn = grid_search.best_estimator_
best_pred = best_knn.predict(X_test)
best_accuracy = accuracy_score(y_test, best_pred)
print(f"Test Accuracy with Best Model: {best_accuracy:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **K-NN Basics**
   - Instance-based (lazy) learning - no explicit training phase
   - Predictions based on k nearest neighbors
   - Simple but effective for local patterns

2. **Distance Metrics**
   - **Euclidean**: Standard straight-line distance
   - **Manhattan**: Sum of absolute differences (city block distance)
   - **Minkowski**: Generalization of both

3. **Key Parameters**
   - **k (n_neighbors)**: Most important - balance between bias and variance
   - **weights**: Uniform (equal) or distance-based (closer neighbors matter more)
   - **metric**: Distance function to use

4. **Best Practices**
   - Always scale features (distance-based algorithm)
   - Use cross-validation to find optimal k
   - Consider distance weighting for better performance
   - Be aware of curse of dimensionality

### When to Use K-Nearest Neighbors

✅ **Good for:**
- Small to medium datasets
- Local patterns in data
- Non-parametric approach needed
- Simple baseline model
- When interpretability is helpful (can show neighbors)

❌ **Not ideal for:**
- Very large datasets (slow prediction)
- High-dimensional data (curse of dimensionality)
- Sparse data
- Real-time predictions on large datasets
- When training data changes frequently

### Next Steps

- Try **Weighted K-NN** for better performance
- Explore **Locally Weighted Regression** (LOWESS)
- Consider **KD-Tree** or **Ball Tree** for faster neighbor search
- Compare with **Decision Trees** for similar use cases
